# 05 — Measure key events

In [ ]:
# Standard project configuration and isolated output namespace
from pathlib import Path
import sys

NOTEBOOK_NAME = "11_measure_key_events.ipynb"
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT_HINT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT_HINT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

from project_config import notebook_context

CTX = notebook_context(NOTEBOOK_NAME, start=CURRENT_DIR)
CONFIG = CTX.config
PROJECT_ROOT = CTX.project_root

# Every notebook writes only inside its own numerically coded namespace.
OUTPUT_DIR = CTX.output_dir
OUTPUT_DATA_DIR = CTX.data_dir
OUTPUT_FIGURE_DIR = CTX.figure_dir
OUTPUT_LOG_DIR = CTX.log_dir

# Backward-compatible aliases used by older cells in this notebook.
DERIVED_OUTPUT_DIR = OUTPUT_DATA_DIR
FIGURE_DIR = OUTPUT_FIGURE_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook output: {OUTPUT_DIR}")


### Workflow contract

- Configuration is loaded from `config/project.yml`.
- This notebook writes only to `11_measure_key_events/` under the configured output root.
- Output filenames carry the `11_` prefix where they are declared explicitly.
- Upstream products are read through the product registry in the YAML file where practical.
- Existing outputs are protected from accidental overwrite by default.


Measure local-baseline-corrected positive, negative, peak-to-peak, and reduced pressure for the key arrivals.

05_measure_key_events.ipynb
Useful but probably too small to remain standalone.
It has only a few cells and writes key-event pressure measurements. These measurements probably belong in one of two places:
* Notebook 10, as a named-event summary;
* Notebook 12, if they are only annotations used in figures.
I would keep it temporarily as:
11_measure_key_events.ipynb
but expect to merge it later.
Verdict: retain for now; likely merge.

In [ ]:

from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime

from project_config import ensure_output_dirs

PATHS = {
    "outputs": CTX.output_dir,
    "derived": CTX.data_dir,
    "figures": CTX.figure_dir,
    "response_correction": CONFIG.product("corrected_waveform_pickle").parent,
}
DERIVED_DIR = CTX.data_dir
FIGURE_DIR = CTX.figure_dir

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [ ]:

from obspy import read
from event_measurements import (
    measure_reduced_pressures_in_window,
    pressure_results_for_paper,
)

st_corr = read(str(CONFIG.product("analysis_window_pickle", required=True)),
               format="PICKLE")
geometry = pd.read_csv(CONFIG.product("geometry_csv", required=True))
SENSOR_DISTANCES_M = (
    geometry.loc[geometry["channel"].str.startswith("HD")]
    .set_index("channel")["distance_m"]
    .to_dict()
)


## Measurement windows

In [ ]:

EXPLOSION_TIME = UTCDateTime("2016-09-01T13:07:12.080")

event_windows = [
    {
        "event": "Initial second-stage failure",
        "start": EXPLOSION_TIME + 3.0 - 0.08,
        "end": EXPLOSION_TIME + 5.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.80,
        "signal_start_s": 0.85,
        "signal_end_s": 1.40,
    },
    {
        "event": "Principal explosion",
        "start": EXPLOSION_TIME + 6.0 - 0.08,
        "end": EXPLOSION_TIME + 9.0 - 0.08,
        "baseline_start_s": 0.10,
        "baseline_end_s": 0.75,
        "signal_start_s": 0.75,
        "signal_end_s": 2.50,
    },
    {
        "event": "Capsule pulse 1",
        "start": UTCDateTime("2016-09-01T13:07:28.30"),
        "end": UTCDateTime("2016-09-01T13:07:28.75"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.12,
        "signal_start_s": 0.12,
        "signal_end_s": 0.45,
    },
    {
        "event": "Capsule pulse 2",
        "start": UTCDateTime("2016-09-01T13:07:28.85"),
        "end": UTCDateTime("2016-09-01T13:07:29.25"),
        "baseline_start_s": 0.00,
        "baseline_end_s": 0.10,
        "signal_start_s": 0.10,
        "signal_end_s": 0.40,
    },
]


In [ ]:

all_results = []
event_streams = {}

for spec in event_windows:
    event_stream, result = measure_reduced_pressures_in_window(
        st_corr,
        spec["start"],
        spec["end"],
        SENSOR_DISTANCES_M,
        reference_distance_m=1000.0,
        event_name=spec["event"],
        baseline_start_s=spec["baseline_start_s"],
        baseline_end_s=spec["baseline_end_s"],
        signal_start_s=spec["signal_start_s"],
        signal_end_s=spec["signal_end_s"],
    )
    event_streams[spec["event"]] = event_stream
    all_results.append(result)

key_event_pressures = pd.concat(all_results, ignore_index=True)
display(pressure_results_for_paper(key_event_pressures))
key_event_pressures.to_csv(
    DERIVED_DIR / "11_key_event_pressure_measurements.csv",
    index=False,
)
